In [0]:
landing_root = "/Volumes/gdelt_dev/raw/gdelt_landing/"

In [0]:
display(dbutils.fs.ls(landing_root))

In [0]:
gkg_landing_path = (
    "/Volumes/gdelt_dev/raw/gdelt_landing/gkg/"
)

display(dbutils.fs.ls(gkg_landing_path))

In [0]:
for item in dbutils.fs.ls(gkg_landing_path):
    print(item.path)

In [0]:
test_date_path = (
"/Volumes/gdelt_dev/raw/gdelt_landing/gkg/ingestion_date=2026-08-07/"
)

display(dbutils.fs.ls(test_date_path))

Create a separate control Volume

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS gdelt_dev.raw.gdelt_control
LOCATION 'abfss://gdelt@stgdeltkartik02.dfs.core.windows.net/control';

In [0]:
gkg_landing_path = (
    "/Volumes/gdelt_dev/raw/gdelt_landing/gkg/"
)

checkpoint_path = (
    "/Volumes/gdelt_dev/raw/gdelt_control/"
    "checkpoints/bronze_gkg/"
)

schema_path = (
    "/Volumes/gdelt_dev/raw/gdelt_control/"
    "schemas/bronze_gkg/"
)

bronze_table = "gdelt_dev.bronze.gdelt_gkg"

In [0]:
sample_df = (
    spark.read
    .option("wholetext", "false")
    .text(gkg_landing_path)
)

display(sample_df.limit(30))

In [0]:
from pyspark.sql import functions as F

sample_split = (
    sample_df
    .select(
        F.size(F.split(F.col("value"), "\t")).alias("column_count")
    )
)

In [0]:
display(sample_split.groupBy("column_count").count().orderBy(F.desc("count")))

In [0]:
sample_split = (
    sample_df
    .select(
        F.split(F.col("value"), "\t"))
    )

In [0]:
display(sample_split)

# **Auto Loader stream**

In [0]:
from pyspark.sql import functions as F


bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "text")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.includeExistingFiles", "false")
    .load(gkg_landing_path)
)

##  Add technical metadata

In [0]:
bronze_enriched = (
    bronze_stream
    .select(
        F.col("value").alias("raw_record"),
        F.col("_metadata.file_name").alias("source_file_name"),
        F.col("_metadata.file_path").alias("source_file_path"),
        F.col("_metadata.file_size").alias("source_file_size"),
        F.col("_metadata.file_modification_time")
            .alias("source_file_modification_time"),
        F.current_timestamp().alias("ingested_at")
    )
)

In [0]:
bronze_enriched = (
    bronze_enriched
    .withColumn(
        "ingestion_date",
        F.regexp_extract(
            F.col("source_file_path"),
            r"ingestion_date=(\d{4}-\d{2}-\d{2})",
            1
        ).cast("date")
    )
)

In [0]:
bronze_enriched.printSchema()

In [0]:
bronze_query = (
    bronze_enriched.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow= True)
    .toTable(bronze_table)
)

In [0]:
spark.streams.active

In [0]:
%sql
SELECT COUNT(*)
FROM gdelt_dev.bronze.gdelt_gkg;

In [0]:
%sql
SELECT *
FROM gdelt_dev.bronze.gdelt_gkg
order by source_file_name desc
limit 1000;

In [0]:
progress = bronze_query.lastProgress

print(progress)